In [45]:
#A copy of Lanczos algorithm on any hermitian matrix
import numpy as np
from copy import deepcopy
from scipy.sparse import random

In [46]:
#generate an arbitrary real symmetric sparse matrix
dim=10000
density=0.1
mat_re=random(dim,dim,density,format='csr')
mat_im=random(dim,dim,density,format='csr')
mat=mat_re+1j*mat_im
mat=(mat+mat.conj().T)*0.5
print(mat.toarray())


[[0.        +0.j         0.7517485 +0.j         0.        +0.j
  ... 0.        +0.j         0.        -0.21741884j
  0.        -0.22083275j]
 [0.7517485 +0.j         0.        +0.j         0.        +0.30201628j
  ... 0.        +0.j         0.        +0.2757098j
  0.48983866-0.07404093j]
 [0.        +0.j         0.        -0.30201628j 0.        +0.j
  ... 0.        -0.10033026j 0.        +0.j
  0.        +0.j        ]
 ...
 [0.        +0.j         0.        +0.j         0.        +0.10033026j
  ... 0.        +0.j         0.31658101+0.013165j
  0.        +0.j        ]
 [0.        +0.21741884j 0.        -0.2757098j  0.        +0.j
  ... 0.31658101-0.013165j   0.        +0.j
  0.        +0.j        ]
 [0.        +0.22083275j 0.48983866+0.07404093j 0.        +0.j
  ... 0.        +0.j         0.        +0.j
  0.22512829+0.j        ]]


In [47]:
#the correct answer to eigen pairs
eigvals,eigvecs=np.linalg.eigh(mat.toarray())
print('eigenvalues are')
print(eigvals)

print('the corresponding eigenvectors are')
print(eigvecs)

eigenvalues are
[-35.05508779 -34.97036746 -34.92147896 ...  34.98349063  35.0673935
 500.73850239]
the corresponding eigenvectors are
[[-4.17727353e-03+0.j          1.24717114e-02+0.j
  -1.80743413e-02+0.j         ...  9.04660812e-03+0.j
  -5.63715942e-03+0.j         -1.00460232e-02+0.j        ]
 [ 1.21499801e-03+0.00077699j -8.23175531e-05-0.0044035j
  -2.65806593e-03-0.00714527j ...  1.41085114e-03-0.00338378j
  -3.99451589e-03-0.00409934j -9.93650738e-03+0.00052472j]
 [-5.65612042e-03+0.00701812j  5.32648750e-05-0.01396301j
   3.06381491e-03-0.00788173j ...  2.57214432e-03-0.00677656j
  -3.89977195e-03-0.00054211j -1.01759946e-02+0.00048884j]
 ...
 [-4.25214196e-03-0.00426389j -3.39467623e-03-0.00996531j
  -3.68417894e-03-0.01194596j ... -3.02771171e-03+0.00266264j
  -4.38851941e-03-0.00600043j -9.93300355e-03+0.00065397j]
 [ 8.36256182e-03-0.0002319j   7.13730937e-03-0.01585411j
  -2.03269262e-03-0.00704821j ... -1.92852967e-03-0.00893046j
  -2.17135399e-03-0.01034391j -9.96713291

In [48]:
# try the lanczos algorithm
# use while loop, set condition by 'happy breakdown' cond
# k= required number of eigen pairs
def Lanczos(mat,k):
    mat=deepcopy(mat.toarray())
    dim=mat.shape[0]
    
    if k>dim:
        k=dim
    T=np.zeros((k,k),dtype=complex)
    V=np.zeros((dim,k),dtype=complex)
    q0=np.random.randn(dim)+1j*np.random.randn(dim)
    q=q0/np.linalg.norm(q0) #first Vrow
    V[:,0]=q
    
    q0=np.zeros(dim,dtype=complex)
    b=0
    
    for i in range(k):
        q=V[:,i]
        v=mat@q
        a=np.vdot(q,v)
        T[i,i]=a
        r=v-a*q
        if i>0:
            r-=b*V[:,i-1]
        b=np.linalg.norm(r)
        if i<k-1:
            if b<1e-12: #'happy breakdown'
                return T[:i+1,:i+1],V[:,:i+1]
            T[i,i+1]=b
            T[i+1,i]=b
            V[:,i+1]=r/b
    
    return T,V
T,V=Lanczos(mat,80)
print(T)

[[-3.22399410e-04+9.71445147e-17j  1.79097555e+01+0.00000000e+00j
   0.00000000e+00+0.00000000e+00j ...  0.00000000e+00+0.00000000e+00j
   0.00000000e+00+0.00000000e+00j  0.00000000e+00+0.00000000e+00j]
 [ 1.79097555e+01+0.00000000e+00j  1.29031061e+01-1.77635684e-15j
   8.13358794e+01+0.00000000e+00j ...  0.00000000e+00+0.00000000e+00j
   0.00000000e+00+0.00000000e+00j  0.00000000e+00+0.00000000e+00j]
 [ 0.00000000e+00+0.00000000e+00j  8.13358794e+01+0.00000000e+00j
   4.64448411e+02+2.55795385e-13j ...  0.00000000e+00+0.00000000e+00j
   0.00000000e+00+0.00000000e+00j  0.00000000e+00+0.00000000e+00j]
 ...
 [ 0.00000000e+00+0.00000000e+00j  0.00000000e+00+0.00000000e+00j
   0.00000000e+00+0.00000000e+00j ...  6.71367811e-01-2.77555756e-16j
   2.73963666e+01+0.00000000e+00j  0.00000000e+00+0.00000000e+00j]
 [ 0.00000000e+00+0.00000000e+00j  0.00000000e+00+0.00000000e+00j
   0.00000000e+00+0.00000000e+00j ...  2.73963666e+01+0.00000000e+00j
   2.97501611e+02+1.70530257e-13j  2.45590479e+

In [49]:
# further diagonalize the matrix T
# using QR factorization with implicit shifts

def Givens(x,y):
    c = abs(x)/np.sqrt(abs(x)**2+abs(y)**2)
    s = np.sign(x)*y/np.sqrt(abs(x)**2+abs(y)**2)
    G=np.array([[c,s],[-s,c]])
    return G

def QR(T):
    dim=T.shape[0]
    Z=np.eye(dim)
    
    T=deepcopy(T)
    T=np.real(T)
    epsilon=10e-6
    
    if dim<=1:
        return T,Z
    it=0

    while True:
        it+=1
        dim=T.shape[0]

        tau=(T[dim-2,dim-2]-T[dim-1,dim-1])/2
        sgn=np.copysign(1.0,tau)
        den=tau+sgn*np.sqrt(tau*tau+T[dim-1,dim-2]**2)
        if den==0:
            mu=T[dim-1,dim-1]
        else:
            mu=T[dim-1,dim-1]-T[dim-1,dim-2]**2/den

        for i in range(0,dim-1):
            if i==0:
                G=Givens(T[0,0]-mu,T[1,0])
            else:
                G=Givens(T[i,i-1],T[i+1,i-1])

            T[i:i+2,:]=G@T[i:i+2,:]
            T[:,i:i+2]=T[:,i:i+2]@G.T
            Z[:,i:i+2]=Z[:,i:i+2]@G.T
            
            

            if abs(T[i,i+1])**2<=epsilon**2*abs(T[i,i]*T[i+1,i+1]):
                T[i,i+1]=T[i+1,i]=0.0
                T11, Z11 = QR(T[0:i+1,0:i+1])
                T22, Z22 = QR(T[i+1:dim, i+1:dim])
                T[0:i+1,0:i+1] = T11
                T[i+1:dim,i+1:dim] = T22
                Z[:,0:i+1] = Z[:,0:i+1] @ Z11
                Z[:,i+1:dim] = Z[:,i+1:dim] @ Z22
                return T,Z

T,Z=QR(T)
eigval=np.diag(T)
print('the eigenvalues are')
print(sorted(eigval))

the eigenvalues are
[np.float64(-35.0503991710905), np.float64(-34.93917086764324), np.float64(-34.80680666059854), np.float64(-34.55318453210823), np.float64(-34.269832733765355), np.float64(-33.92184602929004), np.float64(-33.506118078683684), np.float64(-33.03228491219658), np.float64(-32.492527153611846), np.float64(-31.9295873089974), np.float64(-31.234383848484494), np.float64(-30.551193489860957), np.float64(-29.790291885710936), np.float64(-28.975373012317124), np.float64(-28.108991141888346), np.float64(-27.215111579342114), np.float64(-26.245840712228127), np.float64(-25.191072176746516), np.float64(-24.13261968485055), np.float64(-23.008943847075805), np.float64(-21.85209560277392), np.float64(-20.679878683676023), np.float64(-19.423713214940957), np.float64(-18.144914607740887), np.float64(-16.873238593451497), np.float64(-15.518773595015162), np.float64(-14.16280342333719), np.float64(-12.741376288136827), np.float64(-11.385352412763261), np.float64(-9.917378136816431), np

In [50]:
# combine V and Z to find the eigenvectors
eigvec=V@Z
print('the corresponding eigenvectors are')
print(eigvec)

the corresponding eigenvectors are
[[-0.00163638+9.91185337e-03j -0.00359517+9.38066892e-03j
  -0.00328593-9.49343640e-03j ... -0.01164428-4.96306596e-03j
   0.00254022-1.06943210e-02j  0.00854282-8.53144314e-05j]
 [-0.00110083+9.88926974e-03j -0.00306604+9.46619779e-03j
  -0.00374597-9.21831143e-03j ...  0.01146242-1.28947892e-02j
   0.00541022-1.20755578e-03j -0.00912612-9.25754614e-03j]
 [-0.00117524+1.01197120e-02j -0.00318524+9.67697122e-03j
  -0.0037904 -9.45636392e-03j ...  0.00968312-2.61674493e-04j
   0.0011037 +7.57215748e-03j  0.00143836+6.73517622e-03j]
 ...
 [-0.00097273+9.90686696e-03j -0.00294407+9.50919101e-03j
  -0.00386696-9.17271884e-03j ...  0.00798717-6.60504171e-04j
  -0.00989132+7.13658055e-03j  0.00202089+5.80341694e-03j]
 [-0.00116061+9.91044231e-03j -0.00312885+9.47493414e-03j
  -0.00370351-9.26541180e-03j ... -0.01045018-9.81270645e-04j
   0.00671965+7.33548231e-03j -0.00476807-5.27550026e-03j]
 [-0.00127251+1.01100072e-02j -0.00327857+9.64793626e-03j
  -0.00